# 08 — Topology evolution

**Goal:** let an optimizer redesign the **shape** of your workflow, not just the prompts.

So far we have evolved prompts (notebooks 06 and 07). The orchestration shape was fixed: one agent, one pipeline. But what if the shape itself is wrong? Maybe the right answer is "run two agents in parallel, then a verifier". Or "loop until a check passes". You won't know unless you can search over shapes.

That is what this notebook does. A **meta agent** designs new topologies. Each design is **typed JSON** (`TopologySpec`), not raw Python. The optimizer runs each candidate against a small gold set, scores it, and keeps the winners.

The shape of the loop is:

```
design → reflect → evaluate → archive → repeat
```

This is the ADAS pattern ([arXiv 2408.08435](https://arxiv.org/abs/2408.08435)), with one key difference: we emit **typed specs**, not raw Python. That means no `exec()`, no sandbox needed for the topology itself, and the search is bounded by the registered agents and callables you provide.

**What you will see:**

1. A small gold set with three buckets (factual / inferential / abstain).
2. A baseline single-agent topology that already does well — so the search has limited room to improve.
3. The meta agent proposing topology mutations across 5 generations.
4. The Pareto frontier across accuracy, cost, latency, agent count, and depth.
5. Per-bucket lift compared to the seed.

**Cost:** ~$1–2 for the search.


## 1. Setup — agents and registries

The meta agent is allowed to compose topologies using **only** the agents and callables you register. This is the safety boundary: it cannot invent new agents.

We register three:

- **`cot`** — a chain-of-thought agent. Initial prompt is deliberately weak (no abstain rule), so the search has somewhere to improve.
- **`verifier`** — re-checks the answer against the paragraph.
- **`abstainer`** — knows about the "the paragraph does not say" pattern.


In [1]:
from __future__ import annotations
from typing import Any
from pydantic import BaseModel, Field
from orqest import load_config
from orqest.agents import BaseAgent, GlobalState

config = load_config()
print(f"Task model:    {config.llm_model}")
print(f"Reflection:    {config.llm_model} (overridable via MetaAgentConfig)")


class Answer(BaseModel):
    answer: str = Field(description="Direct answer or 'the paragraph does not say'.")


class CotAgent(BaseAgent[GlobalState, Answer]):
    async def _run_implementation(self, state, **kw):
        latest = state.get_latest_message("user") or ""
        result = await self.call_model(latest, state)
        return result.output


class VerifierAgent(BaseAgent[GlobalState, Answer]):
    async def _run_implementation(self, state, **kw):
        latest = state.get_latest_message("user") or ""
        result = await self.call_model(
            f"Re-check this answer for the question. If wrong, correct it. Question and prior answer:\n{latest}",
            state,
        )
        return result.output


def make_cot():
    return CotAgent(
        agent_name="cot",
        system_prompt="Answer the question. Be concise.",        # weak on purpose
        output_type=Answer,
        model=config.llm_model,
        api_key=config.llm_api_key,
    )


def make_verifier():
    return VerifierAgent(
        agent_name="verifier",
        system_prompt=(
            "You receive a question and a candidate answer. Re-read the paragraph carefully, "
            "verify the answer is supported by the paragraph, and correct it if not. "
            "If the paragraph does not contain the answer, reply 'the paragraph does not say'."
        ),
        output_type=Answer,
        model=config.llm_model,
        api_key=config.llm_api_key,
    )


def make_abstainer():
    return CotAgent(
        agent_name="abstainer",
        system_prompt=(
            "For paragraphs that are off-topic or do not contain the answer, abstain by replying "
            "exactly 'the paragraph does not say'. Otherwise, answer the question concisely."
        ),
        output_type=Answer,
        model=config.llm_model,
        api_key=config.llm_api_key,
    )


agent_registry = {
    "cot": make_cot,
    "verifier": make_verifier,
    "abstainer": make_abstainer,
}
print(f"agent_registry: {sorted(agent_registry)}")

Task model:    openai:gpt-5.2
Reflection:    openai:gpt-5.2 (overridden via MetaAgentConfig if you want a stronger one)
agent_registry: ['abstainer', 'cot', 'verifier']


## 2. The gold set

Same three buckets as notebook 06: **factual**, **inferential**, **abstain**. Nine examples, three per bucket. Small enough to keep the search cheap.

The abstain bucket is the interesting one for topology search: a single weak `cot` agent (without an abstain instruction) will happily hallucinate. A topology that ends with the `abstainer` or with a `verifier` step should do better.


In [2]:
from orqest.optimization import GoldExample


def ex(question: str, paragraph: str, expected: str, bucket: str) -> GoldExample[str, Answer]:
    return GoldExample[str, Answer](
        input=f"Paragraph: {paragraph}\n\nQuestion: {question}",
        expected=Answer(answer=expected),
        id=bucket,
    )


# Factual — answer literally in the paragraph.
factual = [
    ex("In what year was Python first released?",
       "Python was first released in 1991 by Guido van Rossum.", "1991", "factual"),
    ex("Who is the CEO of Tesla?",
       "Tesla, headquartered in Austin, is led by CEO Elon Musk and produces electric vehicles.",
       "Elon Musk", "factual"),
    ex("What is the capital of Germany?",
       "Germany's capital, Berlin, is home to about 3.7 million people.", "Berlin", "factual"),
]

# Inferential — combine two facts.
inferential = [
    ex("How many years between the release of Python and JavaScript?",
       "Python was released in 1991. JavaScript first appeared in 1995.", "4", "inferential"),
    ex("Which is older — the company or its CEO's tenure?",
       "Tesla was founded in 2003. Elon Musk became CEO in 2008.", "the company", "inferential"),
    ex("In which city does the older university operate?",
       "Harvard was founded in 1636 in Cambridge, Massachusetts. MIT was founded in 1861 also in Cambridge.",
       "Cambridge", "inferential"),
]

# Abstain — answer is NOT in the paragraph.
abstain = [
    ex("What is the speed of light in km/s?",
       "This paragraph discusses the history of jazz music in New Orleans during the 1920s.",
       "the paragraph does not say", "abstain"),
    ex("Who painted the Mona Lisa?",
       "Solar panels convert sunlight into electricity at efficiencies typically between 15% and 22%.",
       "the paragraph does not say", "abstain"),
    ex("What is the population of Tokyo?",
       "Coffee originated in Ethiopia and was first cultivated in Yemen during the 15th century.",
       "the paragraph does not say", "abstain"),
]

gold_set = factual + inferential + abstain
print(f"Gold set: {len(gold_set)} examples "
      f"({len(factual)} factual / {len(inferential)} inferential / {len(abstain)} abstain)")

Gold set: 9 examples (3 factual / 3 inferential / 3 abstain)


## 3. Score function and per-bucket helpers

The score function is the same simple substring check we have used before.

The per-bucket helper splits the bundle list by example bucket. This is what lets us see *where* the search improved (or didn't).


In [3]:
def score_fn(output: Answer, example: GoldExample[str, Answer]) -> float:
    """Substring match: 1.0 if expected appears in output, else 0.0."""
    if example.expected is None:
        return 0.0
    out_low = output.answer.lower().strip()
    exp_low = example.expected.answer.lower().strip()
    if exp_low in out_low:
        return 1.0
    if out_low == exp_low:
        return 1.0
    return 0.0


def per_bucket(bundles, examples):
    by_bucket: dict[str, list[float]] = {}
    for b, e in zip(bundles, examples):
        by_bucket.setdefault(e.id, []).append(b.accuracy)
    return {k: sum(v) / len(v) for k, v in by_bucket.items()}


def print_per_bucket(label, bucket_scores):
    print(f"{label}:")
    for bucket in ("factual", "inferential", "abstain"):
        score = bucket_scores.get(bucket)
        print(f"  {bucket:<14} {score:.2f}" if score is not None else f"  {bucket:<14} —")
    overall = sum(bucket_scores.values()) / len(bucket_scores) if bucket_scores else 0.0
    print(f"  {'overall':<14} {overall:.2f}")

## 4. The seed topology

The starting point: a `Pipeline` with one step — the `cot` agent. The simplest possible topology.

We also register one **callable** the meta agent is allowed to use: `is_long`. It's a function the router could use as a condition. The meta agent can compose these names, but cannot invent new ones.


In [4]:
from orqest.orchestration.spec import (
    AgentStepSpec, ParallelSpec, PipelineSpec, PipelineStepEntry, RouterSpec, RouteSpec,
)
from orqest.orchestration.hydrate import CallableRegistry


seed_topology = PipelineSpec(
    steps=[PipelineStepEntry(operation=AgentStepSpec(agent_name="cot"))]
)


# A callable the meta agent can reference by name.
def is_long(input_text):
    return len(str(input_text)) > 200


callable_registry = CallableRegistry()
callable_registry.register("is_long", is_long)

print(f"callable_registry: {callable_registry.names()}")
print(f"\nseed topology JSON:\n{seed_topology.model_dump_json(indent=2)}")

callable_registry: ['is_long']

seed topology JSON:
{
  "kind": "pipeline",
  "steps": [
    {
      "operation": {
        "kind": "agent_step",
        "agent_name": "cot",
        "inline_spec": null
      },
      "config": null
    }
  ],
  "name": "pipeline"
}


## 5. Measure the baseline

Run the seed topology against the gold set. This gives us a number to compare against later.


In [5]:
import asyncio
from orqest.optimization import TopologyEvaluator


evaluator = TopologyEvaluator(
    score_fn=score_fn,
    callable_registry=callable_registry,
    agent_registry=agent_registry,
    topology_gene_name="main",
)

baseline_bundles = await evaluator.evaluate_batch(
    decoded={"main": seed_topology},
    batch=gold_set,
)
baseline_buckets = per_bucket(baseline_bundles, gold_set)
print_per_bucket("BASELINE (single CoT)", baseline_buckets)

BASELINE (single CoT):
  factual        1.00
  inferential    1.00
  abstain        1.00
  overall        1.00


## 6. Run the meta-agent search

This is the heart of the notebook. The search runs for 5 generations. In each generation:

1. **Design** — the meta agent proposes a new topology (a fresh `TopologySpec`).
2. **Reflect** — it reviews its proposal once.
3. **Evaluate** — the proposal is hydrated and run against a minibatch of the gold set.
4. **Archive** — winners are kept.

Two settings worth understanding:

- **`archive_strategy="top_k"`** — keep the K best candidates. This [outperforms](https://arxiv.org/abs/2510.06711) the original ADAS "cumulative" strategy.
- **`reflexion_passes=1`** — one self-review per candidate. More passes = better designs but more cost.

We subscribe to the bus to see the iterations as they happen.


In [6]:
from orqest.optimization import MetaAgentConfig, MetaAgentSearch, TopologyGene
from orqest.observability.events import EventBus


bus = EventBus()
events = []
bus.subscribe("meta_agent.iteration_completed", events.append)


gene = TopologyGene(
    name="main",
    initial=seed_topology,
    constraints=(
        "For abstain examples (off-topic paragraphs), the topology MUST end with an agent "
        "capable of abstaining ('the paragraph does not say'). "
        "Prefer simpler topologies — only add structure if it helps a specific bucket."
    ),
)

search = MetaAgentSearch(
    MetaAgentConfig(
        n_generations=5,
        archive_strategy="top_k",
        archive_size=3,
        reflexion_passes=1,
        debug_max=2,
        minibatch_size=3,
        seed=42,
    ),
    gene=gene,
    evaluator=evaluator,
    meta_agent_model=config.llm_model,
    api_key=config.llm_api_key,
    bus=bus,
)

result = await search.run(trainset=gold_set)

print(f"\nbest_score (aggregate, on minibatch): {result.best_score:.3f}")
print(f"archive size:                         {len(result.raw.entries)}")
print(f"pareto candidates:                    {len(result.pareto_candidates)}")


best_score (aggregate, on minibatch): 0.981
archive size:                         5
pareto candidates:                    1


## 7. Look at the archive

The archive holds the surviving topologies and the meta agent's reasoning (`thought`) for each one. This is your audit trail — you can see what the optimizer tried and what it kept.

The Pareto front shows the trade-offs. A topology with higher accuracy but higher latency is on the front. So is a topology with slightly lower accuracy but far fewer agents.


In [7]:
print("=== Archive (newest → oldest) ===\n")
for entry in reversed(result.raw.entries):
    label = "SEED" if entry.generation == -1 else f"gen-{entry.generation}"
    print(f"--- {label}  score={entry.aggregate_score:.3f} ---")
    if entry.thought and entry.thought != "(seed)":
        thought = entry.thought.replace("\n", " ")
        print(f"  thought: {thought[:200]}")
    print()


print("=== Pareto front (accuracy ↑ / cost ↓ / latency ↓) ===\n")
for entry in result.raw.pareto():
    mean_acc = sum(b.accuracy for b in entry.bundles) / max(1, len(entry.bundles))
    mean_lat = sum(b.latency_ms for b in entry.bundles) / max(1, len(entry.bundles))
    n_agents = entry.bundles[0].raw.get("n_agents", 0) if entry.bundles else 0
    depth = entry.bundles[0].raw.get("depth", 0) if entry.bundles else 0
    label = "SEED" if entry.generation == -1 else f"gen-{entry.generation}"
    print(f"{label:>8}  acc={mean_acc:.2f}  lat={mean_lat:>6.0f}ms  n_agents={n_agents}  depth={depth}")

=== Archive (newest → oldest) ===

--- gen-4  score=0.970 ---
  thought: Hydration failed due to unknown callable state_updater_name. Callable registry only includes 'is_long', so avoid RefinementLoop or use 'is_long' where allowed. Provide a structurally different yet val

--- gen-2  score=0.620 ---
  thought: Hydration failed because Router had no matching route: the 'short' route had condition_name=null which is not treated as default match. Fix by adding fallback_step to router (or provide a condition pr

--- gen-1  score=0.981 ---
  thought: Hydration failed because state_updater_name referenced a non-existent callable. Callable registry only contains 'is_long'. Remove RefinementLoop (or set updater to an allowed callable). We'll replace 

--- gen-0  score=0.943 ---
  thought: Replace unknown state_updater_name with only known callable 'is_long' (acts as a placeholder updater) and simplify router by removing classifier to avoid route-name mismatch; keep structural change vs

--- S

## 8. Apply the winner

`apply_result` shows the diff between the seed topology and the winning topology. By default it is `dry_run=True` — it shows the change but does not commit.


In [8]:
from orqest.optimization import apply_result

topology_registry = {"main": seed_topology}
diffs = apply_result(result, target=topology_registry, dry_run=True)

for d in diffs:
    if d.changed:
        print(f"would change {d.gene_name!r}:")
        print(d.unified[:1000])

would change 'main':
--- main (before)
+++ main (after)
@@ -3,12 +3,84 @@
   "steps": [
     {
       "operation": {
-        "kind": "agent_step",
-        "agent_name": "cot",
-        "inline_spec": null
+        "kind": "router",
+        "routes": [
+          {
+            "name": "complex",
+            "step": {
+              "kind": "pipeline",
+              "steps": [
+                {
+                  "operation": {
+                    "kind": "agent_step",
+                    "agent_name": "cot",
+                    "inline_spec": null
+                  },
+                  "config": null
+                },
+                {
+                  "operation": {
+                    "kind": "agent_step",
+                    "agent_name": "verifier",
+                    "inline_spec": null
+                  },
+                  "config": null
+                }
+              ],
+              "name": "complex_path"
+            },
+            "condition_name":

Now commit the swap and measure per-bucket lift over the full gold set.

In [9]:
apply_result(result, target=topology_registry, dry_run=False)
winning = topology_registry["main"]
print(f"winning topology kind: {winning.kind}")

evolved_bundles = await evaluator.evaluate_batch(
    decoded={"main": winning},
    batch=gold_set,
)
evolved_buckets = per_bucket(evolved_bundles, gold_set)

print()
print_per_bucket("BASELINE", baseline_buckets)
print()
print_per_bucket("EVOLVED", evolved_buckets)
print()
print("Lift per bucket:")
for bucket in ("factual", "inferential", "abstain"):
    delta = evolved_buckets.get(bucket, 0) - baseline_buckets.get(bucket, 0)
    print(f"  {bucket:<14} {delta:+.2f}")

winning topology kind: pipeline



BASELINE:
  factual        1.00
  inferential    1.00
  abstain        1.00
  overall        1.00

EVOLVED:
  factual        1.00
  inferential    1.00
  abstain        1.00
  overall        1.00

Lift per bucket:
  factual        +0.00
  inferential    +0.00
  abstain        +0.00


## Recap

Here is the picture:

1. The optimizer searches over **shapes**, not just prompts.
2. Shapes are emitted as **typed `TopologySpec` JSON**, not raw Python. That means no `exec()`, no sandbox needed at search time.
3. The meta agent can only compose registered agents and callables. This is the safety boundary.
4. Each candidate is **scored on a minibatch**, archived if it wins, and the **Pareto frontier** captures trade-offs across accuracy / cost / latency / agent count / depth.

**One honest note.** If your baseline already scores 100%, topology search has nothing to improve. You will see the wiring work, but no lift. That is exactly what you want — the optimizer is correctly not regressing the baseline. To see real lift, pick a baseline that has measurable headroom.

**What's next:**

- **Notebook 09** — two-phase optimization: first evolve the topology (like this notebook), then evolve the prompts of the winning topology with GEPA. This is the AFlow trick that gets [~5% of GPT-4o cost](https://arxiv.org/abs/2410.10762) — by decoupling the two searches.
- **`docs/concepts/topology_optimization.md`** — full design and rationale.
